# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset on clinicopathological and molecular features of second primary colorectal cancer in cancer survivors, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as attributes, not subscript)
meta = dataset.metadata

print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"License: {meta.license}")

## 2. Data Overview
Explore available record sets and fields in the dataset.

Below, we list all record sets and print their IDs and their fields (with each field's `@id` and column). 

In [ ]:
# List all record sets by @id
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  name: {rs.name if hasattr(rs,'name') else ''}")
    print(f"  description: {rs.description if hasattr(rs,'description') else ''}")
    # List all fields in the record set
    if hasattr(rs, 'fields'):
        print("Fields:")
        for fld in rs.fields:
            print(f"    - field @id: {fld.id}")
            print(f"      name: {fld.name if hasattr(fld,'name') else ''}")
            print(f"      column: {fld.column if hasattr(fld,'column') else ''}")
            print(f"      dataType: {fld.data_type if hasattr(fld,'data_type') else ''}")
    print("")

### Example records preview
You can preview the first 3 records for each record set by referring to their `@id`.

In [ ]:
# Preview a few records from each record set using their @id
for rs in record_sets:
    print(f"Records for record set {rs.id}:")
    try:
        record_it = dataset.records(record_set=rs.id)
        # Show the first 3 records
        for i, rec in enumerate(record_it):
            print(f"  {rec}")
            if i==2:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We identify available record set `@id`s from the overview step and load each into a pandas DataFrame. We use the `@id` for both record set and field references.

In [ ]:
# Collect record set @id's
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded data for record set {record_set_id}: shape = {dataframes[record_set_id].shape}")
        else:
            print(f"No records found for {record_set_id}")
    except Exception as e:
        print(f"Could not load data for {record_set_id}: {e}")

# Select the main/first record set for further analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain record set: {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Examples: filter rows, normalize a numeric field, and group by another field, always referencing fields by their `@id`.

First, we try to detect numeric fields (typically integers or floats) by looking up data types from the schema, and choose one for EDA.

In [ ]:
# Get field @id and data types for main record set
main_rs = None
for rs in record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break

numeric_field_id = None
candidate_numeric_types = ['schema:Integer', 'schema:Number', 'schema:Float', 'schema:Double']
candidate_groupby_id = None

if main_rs and hasattr(main_rs, 'fields'):
    for fld in main_rs.fields:
        # Find first numeric field
        dt = getattr(fld, 'data_type', None)
        if dt and dt in candidate_numeric_types and numeric_field_id is None:
            numeric_field_id = fld.id
        # Find a non-numeric field for grouping
        elif dt and dt == 'schema:Text' and candidate_groupby_id is None:
            candidate_groupby_id = fld.id

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group-by field: {candidate_groupby_id}")

df = dataframes[main_record_set_id]

if numeric_field_id in df.columns:
    # Remove missing/non-numeric values
    series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = series.mean() # Use mean as threshold example
    filtered_df = df[series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (series - series.mean()) / series.std()
    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field (if exists)
    group_field = candidate_groupby_id
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"\nMean of {numeric_field_id} grouped by {group_field} (using @id):")
        print(grouped_df.head())
else:
    print(f"No numeric field {numeric_field_id} found in dataframe columns.")

## 5. Visualization
Visualize distributions or relationships, referencing columns by their field `@id`.

Here, we show a histogram of the selected numeric field, and (if dataset allows) a boxplot grouped by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=15)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Histogram of {numeric_field_id}')
    plt.show()

    # Boxplot grouped by group_field if possible
    if candidate_groupby_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[candidate_groupby_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.xlabel(candidate_groupby_id)
        plt.ylabel(numeric_field_id)
        plt.title(f'Boxplot of {numeric_field_id} grouped by {candidate_groupby_id}')
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

- This notebook demonstrates the use of the `mlcroissant` library for exploring and processing tabular health/biomedical datasets with a Croissant JSON-LD schema, referencing entities by their `@id`s.
- You have seen how to load dataset metadata, overview available record sets and fields (with IDs), extract and analyze records in pandas, and visualize data.

**For further exploration:** You can filter or group using other fields by their `@id`, enrich visualizations, or incorporate additional `mlcroissant` APIs for advanced semantic queries.